# Assignment 1: Market Basket Analysis using Apriori Algorithm

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jits77111-source/DWBDA-Assignment-1-Market-Basket-Analysis/blob/main/Assignment_1_Market_Basket_Analysis.ipynb)

**Author:** Jit Sarkar  
**Course:** Data Warehousing, Big Data & Analytics (DWBDA)  
**GitHub Repository:** [DWBDA-Assignment-1-Market-Basket-Analysis](https://github.com/jits77111-source/DWBDA-Assignment-1-Market-Basket-Analysis)

---

### Objective
Perform Market Basket Analysis on a retail transaction dataset using the **Apriori Algorithm** to extract frequent itemsets and derive actionable association rules governed by metrics such as **Support**, **Confidence**, and **Lift**.

## 1. Install Dependencies & Import Libraries

In [ ]:
# Install mlxtend if running on Colab or fresh environment
!pip install -q mlxtend pandas matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

print("Libraries successfully imported!")

## 2. Load the Dataset

We use a realistic retail dataset of **Computer & Workstation Accessories Transactions** containing 30 transaction baskets across customers.

In [ ]:
# Download dataset directly from GitHub repo if not already present locally
import os, urllib.request

csv_path = "computer_accessories_transactions.csv"
if not os.path.exists(csv_path):
    raw_url = "https://raw.githubusercontent.com/jits77111-source/DWBDA-Assignment-1-Market-Basket-Analysis/main/computer_accessories_transactions.csv"
    urllib.request.urlretrieve(raw_url, csv_path)
    print(f"Downloaded {csv_path} from GitHub repository!")

df = pd.read_csv(csv_path)
print(f"Dataset Shape: {df.shape}")
df.head(10)

## 3. Preprocess Transactions (One-Hot Encoding)

Transform the transaction item strings into a binary boolean matrix required by the Apriori algorithm using `TransactionEncoder`.

In [ ]:
transactions = [items.split(";") for items in df["Items"]]

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print(f"Encoded Matrix Shape: {df_encoded.shape}")
print(f"Distinct Items ({len(te.columns_)}):", list(te.columns_))
df_encoded.head()

## 4. Mine Frequent Itemsets using Apriori

We define minimum support threshold $\text{min\_support} = 0.20$ (present in at least 20% of all baskets).

In [ ]:
frequent_itemsets = apriori(df_encoded, min_support=0.20, use_colnames=True)
frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(lambda x: len(x))
frequent_itemsets = frequent_itemsets.sort_values(by="support", ascending=False).reset_index(drop=True)

print(f"Total Frequent Itemsets Found: {len(frequent_itemsets)}")
frequent_itemsets

## 5. Generate Association Rules

We set a minimum confidence threshold $\text{min\_confidence} = 0.50$ (50%) and compute key metrics:
- **Support**: Frequency of itemset occurrence.
- **Confidence**: $P(B | A) = \frac{\text{Support}(A \cup B)}{\text{Support}(A)}$
- **Lift**: $\frac{P(B | A)}{P(B)}$ (Ratio $> 1$ indicates positive correlation).
- **Leverage & Conviction**: Measure deviation from statistical independence.

In [ ]:
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.50)
rules = rules.sort_values(by="lift", ascending=False).reset_index(drop=True)

# Format sets for readable display
rules["antecedents_str"] = rules["antecedents"].apply(lambda x: ", ".join(list(x)))
rules["consequents_str"] = rules["consequents"].apply(lambda x: ", ".join(list(x)))

display_cols = ["antecedents_str", "consequents_str", "support", "confidence", "lift", "leverage", "conviction"]
rules[display_cols]

## 6. Visualizing Market Basket Insights

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Support vs Confidence with Lift Heatmap
scatter = axes[0].scatter(
    rules["support"],
    rules["confidence"],
    c=rules["lift"],
    cmap="viridis",
    s=140,
    alpha=0.9,
    edgecolors="black"
)
cbar = fig.colorbar(scatter, ax=axes[0])
cbar.set_label("Lift Metric", fontsize=12)
axes[0].set_title("Support vs. Confidence (Colored by Lift)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Support", fontsize=12)
axes[0].set_ylabel("Confidence", fontsize=12)
axes[0].grid(True, linestyle="--", alpha=0.5)

# 2. Top Association Rules by Lift
top_rules = rules.head(8).copy()
rule_labels = [f"{r['antecedents_str']} -> {r['consequents_str']}" for _, r in top_rules.iterrows()]
axes[1].barh(rule_labels, top_rules["lift"], color="#1f77b4", edgecolor="black")
axes[1].set_xlabel("Lift Value", fontsize=12)
axes[1].set_title("Top 8 Association Rules Ranked by Lift", fontsize=14, fontweight="bold")
axes[1].invert_yaxis()
axes[1].grid(True, axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("mba_visualization.png", dpi=300)
plt.show()

## 7. Business Interpretation & Conclusion

1. **High Affinity Itemsets**:
   - Customers purchasing `{Mechanical Keyboard, Gaming Mouse}` exhibit high confidence and lift (> 2.0) towards purchasing `{Mousepad XL}`.
   - Customers buying `{Laptop Stand, USB-C Hub}` strongly associate with `{HDMI Cable}`.
2. **Strategic Recommendations**:
   - Bundle popular accessories as discounted ergonomic/productivity packages.
   - Cross-sell complementary peripherals during checkout.